# Measure CLIP image-text alignment

This is used to quantify how much an image corresponds to an outdoor scene. In this notebook, we compute CLIP alignment scores of an image with different CLIP models. We then average the alignment scores for each image based on the scores returned by different CLIP model variants.

# Outputs
The outputs of this notebook are stored in [pickle_files](./pickle_files/).

In [ ]:
import torch
from PIL import Image
import open_clip
from pathlib import Path
import os
import pickle
from glob import glob
from tqdm import tqdm

## Model configurations

- 'ViT-L-14', 'laion2b_s32b_b82k'
- 'ViT-B-32', 'laion2b_s34b_b79k'
- 'convnext_base', 'laion400m_s13b_b51k'

## Setup

In [ ]:
os.makedirs("./pickle_files", exist_ok=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

**NOTE**: Update this path accordingly

In [ ]:
base_dir = "path_to_filtered_instacities_dataset"  # Update this path accordingly


## Running VIT-L-14

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="laion2b_s32b_b82k"
)
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer("ViT-L-14")
model.to(device)

In [ ]:
for city_name in os.listdir(base_dir):
    print(f"Processing {city_name}...")
    all_scores = {}

    # Load images
    all_images = glob(f"{os.path.join(base_dir, city_name)}/*.jpg")

    # Compute text embeddings
    text = tokenizer(["an outdoor photo"]).to(device)
    text_features = model.encode_text(text).to(device)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Compute and store embeddings
    for image_path in tqdm(all_images):
        score_key = Path(image_path).stem
        with torch.no_grad(), torch.autocast(device):
            image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
            image_features = model.encode_image(image).to(device)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_probs = float((image_features @ text_features.T).detach().cpu()[0][0])
            all_scores[score_key] = text_probs

    # Store scores
    with open(f"./pickle_files/vitl_{city_name}.pkl", "wb") as f:
        pickle.dump(all_scores, f)


## Running VIT-B-32

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model.to(device)

In [ ]:
for city_name in os.listdir(base_dir):
    print(f"Processing {city_name}...")
    all_scores = {}

    # Load images
    all_images = glob(f"{os.path.join(base_dir, city_name)}/*.jpg")

    # Compute text embeddings
    text = tokenizer(["an outdoor photo"]).to(device)
    text_features = model.encode_text(text).to(device)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Compute and store embeddings
    for image_path in tqdm(all_images):
        score_key = Path(image_path).stem
        with torch.no_grad(), torch.autocast(device):
            image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
            image_features = model.encode_image(image).to(device)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_probs = float((image_features @ text_features.T).detach().cpu()[0][0])
            all_scores[score_key] = text_probs

    # Store scores
    with open(f"./pickle_files/vitb_{city_name}.pkl", "wb") as f:
        pickle.dump(all_scores, f)


## Running ConvNext-Base

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "convnext_base", pretrained="laion400m_s13b_b51k"
)
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer("convnext_base")
model.to(device)

In [ ]:
for city_name in os.listdir(base_dir):
    print(f"Processing {city_name}...")
    all_scores = {}

    # Load images
    all_images = glob(f"{os.path.join(base_dir, city_name)}/*.jpg")

    # Compute text embeddings
    text = tokenizer(["an outdoor photo"]).to(device)
    text_features = model.encode_text(text).to(device)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Compute and store embeddings
    for image_path in tqdm(all_images):
        score_key = Path(image_path).stem
        with torch.no_grad(), torch.autocast(device):
            image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
            image_features = model.encode_image(image).to(device)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_probs = float((image_features @ text_features.T).detach().cpu()[0][0])
            all_scores[score_key] = text_probs

    # Store scores
    with open(f"./pickle_files/convnextb_{city_name}.pkl", "wb") as f:
        pickle.dump(all_scores, f)
